In [1]:
# install sentence-transformers if not already there
import subprocess, sys
try:
    import sentence_transformers
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"])
    import sentence_transformers

In [2]:
import pandas as pd
import numpy as np
import torch
import re
import time
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("torch cuda available:", torch.cuda.is_available())

torch cuda available: True


In [3]:
# load data
COMP = "/kaggle/input/competitions/smart-mcq-solver-challenge"
train = pd.read_csv(f"{COMP}/train.csv")
test = pd.read_csv(f"{COMP}/test.csv")
sample = pd.read_csv(f"{COMP}/sample_submission.csv")
OPTIONS = ["A", "B", "C", "D", "E"]
print(f"train: {train.shape}, test: {test.shape}")
train.head(2)

train: (2000, 8), test: (500, 7)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A


## Corpus Build

For each train row I keep the question text and the correct answer text. Retrieval is done on the question text only. The correct answer text is kept as context to show the model.

In [4]:
# build one document per training row
train_docs = []
for _, r in train.iterrows():
    correct_text = str(r[r["answer"]])
    train_docs.append({
        "prompt": str(r["prompt"]),
        "answer_letter": r["answer"],
        "correct_text": correct_text,
    })

print(f"built {len(train_docs)} training documents")
print()
print("sample doc 0:")
print("  Q:", train_docs[0]["prompt"][:120])
print("  A:", train_docs[0]["correct_text"][:120])

built 2000 training documents

sample doc 0:
  Q: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? amo
  A: Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is 


In [5]:
# load sentence-transformer for embeddings
from sentence_transformers import SentenceTransformer
device = "cuda" if torch.cuda.is_available() else "cpu"
emb_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)
print("loaded MiniLM on", device)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

loaded MiniLM on cuda


In [6]:
# encode training prompts
train_prompts = [d["prompt"] for d in train_docs]

t0 = time.time()
train_embs = emb_model.encode(
    train_prompts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)
print(f"train embeddings shape: {train_embs.shape} (took {time.time()-t0:.1f}s)")

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

train embeddings shape: (2000, 384) (took 1.5s)


In [7]:
# encode test prompts
test_prompts = test["prompt"].astype(str).tolist()

t0 = time.time()
test_embs = emb_model.encode(
    test_prompts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)
print(f"test embeddings shape: {test_embs.shape} (took {time.time()-t0:.1f}s)")

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

test embeddings shape: (500, 384) (took 0.2s)


In [8]:
# retrieve top-k nearest train examples per test row
K = 3
sims = test_embs @ train_embs.T
top_k = np.argsort(-sims, axis=1)[:, :K]
print(f"top-{K} retrieval done, shape {top_k.shape}")
print(f"mean top-1 sim: {sims[np.arange(len(test)), top_k[:,0]].mean():.3f}")

top-3 retrieval done, shape (500, 3)
mean top-1 sim: 0.986


In [9]:
# quick sanity check on retrieval quality
for i in [0, 1, 2]:
    print(f"\ntest row {i}:")
    print("  Q:", test_prompts[i][:120])
    for j, tid in enumerate(top_k[i]):
        print(f"  match {j+1} (sim={sims[i,tid]:.3f}): {train_docs[tid]['prompt'][:100]}")


test row 0:
  Q: Pick the best possible answer: What is the relationship between the Hamiltonians and eigenstates in supersymmetric quant
  match 1 (sim=1.000): Pick the best possible answer: What is the relationship between the Hamiltonians and eigenstates in 
  match 2 (sim=1.000): Pick the best possible answer: What is the relationship between the Hamiltonians and eigenstates in 
  match 3 (sim=0.993): Pick the best possible answer: What is the relationship between the Hamiltonians and eigenstates in 

test row 1:
  Q: What is the estimated redshift of CEERS-93316, a candidate high-redshift galaxy observed by the James Webb Space Telesco
  match 1 (sim=1.000): What is the estimated redshift of CEERS-93316, a candidate high-redshift galaxy observed by the Jame
  match 2 (sim=0.919): Pick the best possible answer: What is the estimated redshift of CEERS-93316, a candidate high-redsh
  match 3 (sim=0.915): Determine the correct option: What is the estimated redshift of CEERS-93316, a 

## LLM inference using context

Building a prompt that puts the retrieved similar questions (with their correct answers) in the context, then asks the model to answer the test question. Model outputs a single letter.

Using Qwen 2.5 3B instead of 1.5B - the smaller one was making basic mistakes in the last version.

In [10]:
# load the language model
from transformers import AutoTokenizer, AutoModelForCausalLM

llm_name = "Qwen/Qwen2.5-3B-Instruct"
tok = AutoTokenizer.from_pretrained(llm_name)
llm = AutoModelForCausalLM.from_pretrained(
    llm_name,
    torch_dtype=torch.float16,
    device_map="auto",
)
llm.eval()
print("loaded", llm_name)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

loaded Qwen/Qwen2.5-3B-Instruct


In [11]:
# build the RAG prompt and get an answer letter
def build_prompt(test_row, retrieved_ids):
    ctx = ""
    for k, tid in enumerate(retrieved_ids):
        d = train_docs[tid]
        ctx += f"Example {k+1}:\n"
        ctx += f"Question: {d['prompt']}\n"
        ctx += f"Correct answer: {d['correct_text']}\n\n"
    opts = "\n".join([f"{L}. {test_row[L]}" for L in OPTIONS])
    prompt = (
        f"Here are some similar solved questions for reference:\n\n{ctx}"
        f"Now answer this question by picking the correct option.\n\n"
        f"Question: {test_row['prompt']}\n\n"
        f"Options:\n{opts}\n\n"
        f"Reply with just the single letter (A, B, C, D or E):"
    )
    return prompt

def parse_letter(text):
    m = re.search(r"[A-E]", str(text))
    return m.group(0) if m else None

def answer_row(test_row, retrieved_ids):
    prompt = build_prompt(test_row, retrieved_ids)
    inputs = tok(prompt, return_tensors="pt", truncation=True, max_length=2048).to(llm.device)
    with torch.no_grad():
        out = llm.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False,
            pad_token_id=tok.eos_token_id,
        )
    gen = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    letter = parse_letter(gen)
    return letter, gen

In [12]:
# quick test on the first few test rows
for i in range(3):
    row = test.iloc[i]
    ret = top_k[i]
    letter, raw = answer_row(row, ret)
    print(f"row {i}: model said {raw.strip()!r} -> {letter}")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


row 0: model said 'A\nYou are an' -> A
row 1: model said 'B' -> B
row 2: model said 'B' -> B


In [13]:
# run RAG over the full test set
predictions = []
malformed = 0

t0 = time.time()
for i in range(len(test)):
    row = test.iloc[i]
    ret = top_k[i]
    try:
        letter, _ = answer_row(row, ret)
    except Exception as e:
        print(f"row {i} failed:", e)
        letter = None
    if letter is None:
        letter = "B"  # most common letter in train, safe fallback
        malformed += 1
    predictions.append(letter)
    if (i+1) % 50 == 0:
        elapsed = time.time() - t0
        eta = elapsed / (i+1) * (len(test) - i - 1)
        print(f"done {i+1}/{len(test)} - elapsed {elapsed:.0f}s eta {eta:.0f}s")

print(f"\ntotal time: {time.time()-t0:.0f}s")
print(f"malformed / fallback: {malformed}")
print(f"rank-1 letter distribution: {dict(Counter(predictions))}")

done 50/500 - elapsed 15s eta 138s
done 100/500 - elapsed 29s eta 117s
done 150/500 - elapsed 43s eta 100s
done 200/500 - elapsed 58s eta 88s
done 250/500 - elapsed 75s eta 75s
done 300/500 - elapsed 90s eta 60s
done 350/500 - elapsed 105s eta 45s
done 400/500 - elapsed 120s eta 30s
done 450/500 - elapsed 136s eta 15s
done 500/500 - elapsed 153s eta 0s

total time: 153s
malformed / fallback: 0
rank-1 letter distribution: {'A': 84, 'B': 112, 'E': 86, 'C': 121, 'D': 97}


## Building rank-2 and rank-3

The competition scores top-3 predictions with MAP@3. Just having rank-1 correct gets you 1.0, rank-2 correct gets 0.5, rank-3 correct gets 1/3.

For rank-2 and rank-3 I use tf-idf cosine similarity between the picked rank-1 text and the other 4 option texts. The idea: if a distractor is similar to the correct answer, it might be a paraphrase that also gets credit, or at least is more "answer-shaped" than the other distractors.

In [14]:
# fit tf-idf on all option texts and prompts from both splits
corpus = []
for L in OPTIONS:
    corpus.extend(train[L].astype(str).tolist())
    corpus.extend(test[L].astype(str).tolist())
corpus.extend(train["prompt"].astype(str).tolist())
corpus.extend(test["prompt"].astype(str).tolist())

vec = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=1,
    max_features=50_000,
    sublinear_tf=True,
    strip_accents="unicode",
)
vec.fit(corpus)
print(f"tf-idf vocab size: {len(vec.vocabulary_):,}")

tf-idf vocab size: 12,749


In [15]:
# build top-3: rank-1 = LLM pick, rank-2/3 = closest other options by tf-idf
top3 = []
for i, r1 in enumerate(predictions):
    row = test.iloc[i]
    r1_text = str(row[r1])
    rest = [L for L in OPTIONS if L != r1]
    rest_texts = [str(row[L]) for L in rest]
    q = vec.transform([r1_text])
    C = vec.transform(rest_texts)
    sim = cosine_similarity(q, C).ravel()
    order = np.argsort(-sim)
    top3.append([r1, rest[order[0]], rest[order[1]]])

# check the first few
for i in range(5):
    print(f"row {i}: {' '.join(top3[i])}")

row 0: A D B
row 1: B A C
row 2: B E D
row 3: E C A
row 4: C A D


In [16]:
# build the submission csv
sub = pd.DataFrame({
    "ID": test["id"],
    "Prediction": [" ".join(t) for t in top3]
})

# sanity checks
assert len(sub) == len(test), f"got {len(sub)} rows, expected {len(test)}"
assert (sub["ID"].values == sample["ID"].values).all(), "ID order mismatch"
assert sub["Prediction"].str.split().apply(len).eq(3).all(), "not all rows have 3 labels"
assert sub["Prediction"].str.split().apply(
    lambda x: all(L in OPTIONS for L in x) and len(set(x)) == 3
).all(), "invalid letter or duplicate in prediction"

sub.to_csv("/kaggle/working/submission.csv", index=False)
print(f"saved submission.csv with {len(sub)} rows")
sub.head(10)

saved submission.csv with 500 rows


,ID,Prediction
0,1,A D B
1,2,B A C
2,3,B E D
3,4,E C A
4,5,C A D
5,6,D E A
6,7,E D C
7,8,B E A
8,9,C D E
9,10,B E A
